## ATNLP Lab 3 – Solving QA with a Sentence Encoder

In this lab, you build a multiple-choice QA baseline for MMLU **without** using an autoregressive language model.

What this means in practice:
1. Convert the question and each answer option into vector embeddings.
2. Compute semantic similarity between the question vector and each option vector.
3. Choose the option with the **highest** cosine similarity.

This gives a fast, transparent baseline that helps you understand retrieval-style reasoning before using larger generative models.

We use the sentence-transformers bi-encoder `all-MiniLM-L6-v2`.

### 1) Setup

Install the required packages (run once per environment), then import all dependencies used in the lab.

`sentence-transformers` provides the encoders, `datasets` loads MMLU, and `tqdm` gives progress bars for evaluation.

In [ ]:
!pip -q install sentence-transformers datasets tqdm

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

### 2) Load data and encoder

First, load a subset of MMLU and initialize the sentence encoder.

We use `high_school_biology` with `test[:200]` so the experiment runs quickly on CPU while still giving enough samples for a meaningful baseline estimate.

Model card: [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

Tip: each MMLU example contains a question, four choices, and the correct answer index.

In [ ]:
task   = "high_school_biology"
mmlu   = load_dataset("cais/mmlu", task, split="test[:200]") 

MODEL_NAME = "all-MiniLM-L6-v2"          # 64 MB, 384-dim vectors
embedder   = SentenceTransformer(MODEL_NAME, device="cpu")

### 3) Implement the embedding baseline

Complete the two functions below.

For each question, the pipeline is:
1. Embed the question.
2. Embed the four answer options.
3. Compute cosine similarity between the question and each option.
4. Return the index of the option with the **highest** score.

Because vectors are L2-normalized, cosine similarity is equivalent to the dot product used in the code.

In [ ]:
def embed(texts, batch_size=32):
    pass

def embed_predict_single(sample, q_emb=None):
    """
    Given one MMLU dict with fields:
      question : str
      choices  : list[str]  (len==4)
    return index 0-3 of predicted answer.
    """
    pass


### 4) Run evaluation

Now run the evaluation loop over the dataset.

For each sample:
1. Predict an answer index with your baseline function.
2. Compare it with the gold answer index.
3. Accumulate counts and compute final accuracy (`correct / total`).

Keep the evaluation deterministic and report the final score clearly.

In [ ]:
for sample in tqdm(mmlu, desc="Predicting answers"):
    pass

### 5) Extension: reformulate MCQA as NLI

This embedding baseline is intentionally simple: a general-purpose bi-encoder plus cosine similarity.

A stronger alternative is to reframe each `(question, choice)` pair as a Natural Language Inference (NLI) input:
- premise: the question context
- hypothesis: the candidate answer

The NLI model outputs probabilities for `contradiction`, `neutral`, and `entailment`. For QA, select the option with the highest **entailment** probability.

Model card: [cross-encoder/nli-deberta-v3-base](https://huggingface.co/cross-encoder/nli-deberta-v3-base)

#### 5.1 Load the NLI model

Initialize the cross-encoder once, then reuse it for all samples.

This model is heavier than the bi-encoder baseline, so single initialization avoids repeated loading overhead.

In [ ]:
from sentence_transformers import CrossEncoder
model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

#### 5.2 Implement NLI prediction

Complete `nli_predict_single(...)` so it:
1. Builds four `(question, answer)` pairs.
2. Runs the NLI model on all four pairs.
3. Extracts the entailment score for each option.
4. Returns the index with the highest entailment probability.

Then evaluate this NLI approach on the same subset and compare it with the embedding baseline.

In [ ]:
def make_pair(question:str, choice:str) -> str:
    return (question, f"Answer: {choice}")

def nli_predict_single(sample, model) -> int:
    """
    Returns index (0-3) of selected answer.
    """
    pass